# BERT 分词与词性联合标注

复用根目录的train.py和main.py。先完成README中的数据准备；从根目录或notebooks启动均可。参数对齐记录的基线：lr3e-5、有效批次64、6轮、seed42。新实验使用独立输出目录。


In [ ]:
from pathlib import Path
import os
import sys

# Notebook可能从子目录启动，统一定位项目根目录后再导入模块。
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "train.py").is_file() and (p / "main.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("请在PKU-SEGPOS项目目录或notebooks子目录中启动Notebook")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 导入业务模块
from train import run_training
DATA_DIR = Path("data/prepared_v3")
assert (DATA_DIR / "manifest.json").is_file(), "请先生成prepared_v3数据版本"


In [ ]:
training_args = [
    "--train", str(DATA_DIR / "train.jsonline"),
    "--dev", str(DATA_DIR / "dev.jsonline"),
    "--catalog", str(DATA_DIR / "label_catalog.json"),
    "--output-dir", "outputs/bert-prepared-v3-lr3e5-seed42",
    "--learning-rate", "3e-5",
    "--epochs", "6",
    "--batch-size", "16",
    "--gradient-accumulation", "4",
    "--max-len", "256",
    "--overlap", "64",
    "--decoder", "viterbi",
    "--seed", "42",
]
# 离线已有预训练缓存时，可添加 --local-files-only。
# 中断恢复只使用本实验目录的检查点；不要恢复旧89标签模型。
# 对照实验只改待研究参数，并使用新的输出目录。


In [ ]:
trainer = run_training(training_args)


最佳模型与训练报告保存在所指定的实验输出目录。完整官方dev用于选模；补充诊断集 `data/prepared_v3/diagnostic_dev.jsonline` 用 `main.py`、`eval.py` 单独评测。人工排查使用同目录的 `testmodel.ipynb`，显式选择对应实验的best_model。旧89标签检查点不能恢复为87标签新实验。
